# 🧪 W6-D7 复习实验：用一个 mini ReAct Agent 串起本周全部机制

> 配套阅读：`ima/第6周-Day7-第六周总复习.md`（知识全景图、综合测试题、误区总汇在那边）
> 复习日不抄笔记，用可执行实验自检四个机制是否真正理解：
>
> 1. **ReAct 状态机**（Thought→Action→Observation）能否完成需要两跳工具调用的任务？
> 2. 单轮直答（没有循环）为什么做不了多跳问题？
> 3. `max_steps` + 降级如何防死循环？
> 4. 自洽性投票（Day4 复习）为什么比单次采样准？
>
> 全部纯 Python/numpy 模拟，无 LLM API。

## 实验 1：Mini ReAct Agent 状态机——跑通"广州和杭州今天相差几度？"

这个任务一跳做不完：先查广州、再查杭州、再相减、才能回答。
ReAct 的每一圈 = Thought（还缺什么信息）→ Action（调工具）→ Observation（结果入上下文），
循环直到 Thought 判定"信息齐了"→ Final。

In [ ]:
WEATHER7 = {"北京": 32, "上海": 28, "广州": 35, "杭州": 30}

def get_weather(city: str):
    """工具 1：查温度"""
    return WEATHER7[city]

def calculate(expr: str):
    """工具 2：算术"""
    return eval(expr)

TOOLS7 = {"get_weather": get_weather, "calculate": calculate}

class MiniReActAgent:
    """ReAct 状态机：THINK → ACT → OBSERVE → (循环) → FINAL"""

    def __init__(self, tools, max_steps=6):
        self.tools, self.max_steps = tools, max_steps

    def thought(self, task, obs):
        """规则版大脑：根据已有观察判断下一步（生产版 = LLM）"""
        cities = [c for c in WEATHER7 if c in task]
        temps = {o["city"]: o["temp"] for o in obs if "city" in o}
        if any(c not in temps for c in cities):
            next_city = next(c for c in cities if c not in temps)
            return {"type": "action", "tool": "get_weather", "args": {"city": next_city},
                    "why": f"还缺 {next_city} 的温度"}
        if not any("diff" in o for o in obs):
            expr = f"{temps[cities[0]]}-{temps[cities[1]]}"
            return {"type": "action", "tool": "calculate", "args": {"expr": expr},
                    "why": "两个温度都到手，做差"}
        return {"type": "final"}

    def run(self, task):
        print(f"🎯 任务: {task}")
        obs = []
        for step in range(1, self.max_steps + 1):
            th = self.thought(task, obs)
            if th["type"] == "final":
                diff = next(o["diff"] for o in obs if "diff" in o)
                print(f"  Thought #{step}: 信息齐全，输出最终答案 ✅")
                return f"两地相差 {abs(diff)}°C"
            print(f"  Thought #{step}: {th['why']}")
            print(f"  Action   : {th['tool']}({th['args']})")
            result = self.tools[th["tool"]](**th["args"])
            if th["tool"] == "get_weather":
                obs.append({"city": th["args"]["city"], "temp": result})
            else:
                obs.append({"diff": result})
            print(f"  Observe  : {result}")
        return "步数耗尽，降级处理"

agent = MiniReActAgent(TOOLS7)
answer = agent.run("广州和杭州今天相差几度？")
print(f"\n✅ 最终答案: {answer}")

## 实验 2：为什么必须有循环——单轮直答 vs ReAct 在多跳任务上的对决

一个"单轮 LLM"（模拟无 Agent 结构的 ChatGPT 式调用）只有一次工具机会就作答。
对需要两跳的任务，它必然缺信息；ReAct 靠循环逐跳补齐。跑 4 个多跳任务统计准确率。

In [ ]:
TASKS = [
    ("广州和杭州相差几度？",   ["广州", "杭州"]),
    ("北京比上海热几度？",     ["北京", "上海"]),
    ("杭州和上海哪个更热？",   ["杭州", "上海"]),
    ("北京和广州相差几度？",   ["北京", "广州"]),
]

def direct_answer(task, cities):
    """单轮模式：只允许一次工具调用就作答——必然缺另一半信息"""
    one_temp = get_weather(cities[0])              # 单轮只允许这一次工具机会
    return None                                   # 第二个城市的温度无从得知

def react_answer(task, cities):
    """ReAct 模式：静默跑实验 1 的状态机，逐跳补齐信息"""
    obs = []
    silent = MiniReActAgent(TOOLS7)
    for _ in range(silent.max_steps):
        th = silent.thought(task, obs)
        if th["type"] == "final":
            return abs(next(o["diff"] for o in obs if "diff" in o))
        r = silent.tools[th["tool"]](**th["args"])
        obs.append({"city": th["args"]["city"], "temp": r} if th["tool"] == "get_weather"
                   else {"diff": r})
    return None

print(f"{'任务':<16s} {'单轮直答':<8s} {'ReAct':<10s}")
d_score = r_score = 0
for task, cities in TASKS:
    d = direct_answer(task, cities)
    truth = abs(WEATHER7[cities[0]] - WEATHER7[cities[1]])
    r = react_answer(task, cities)
    d_ok, r_ok = d == truth, r == truth
    d_score += d_ok; r_score += r_ok
    print(f"{task:<18s} {'✅' if d_ok else '❌ 缺信息':<10s} {'✅ ' + str(r) + '°C' if r_ok else '❌':<8s}")
print(f"\n准确率：单轮直答 {d_score}/{len(TASKS)} vs ReAct {r_score}/{len(TASKS)}")
print("结论：多跳问题的解法不在更好的 prompt，而在『循环 + 观察』这个结构本身。")

## 实验 3：死循环保护——max_steps + 优雅降级

ReAct 是循环，循环就有转不停的风险：工具一直失败、大脑一直重试。
生产 Agent 必须有 `max_steps` 硬上限 + 降级路径。模拟一个宕机的库存 API，
看"没有保护"和"有保护"的差别。

In [ ]:
def check_inventory(item: str):
    raise TimeoutError("库存服务宕机中")     # 永久故障

class GuardedReActAgent(MiniReActAgent):
    """加了护栏的 ReAct：步数上限 + 失败降级 + 参数错误不重试"""

    def run(self, task):
        print(f"🎯 任务: {task}（库存 API 已宕机）")
        obs, failures = [], 0
        for step in range(1, self.max_steps + 1):
            th = {"type": "action", "tool": "check_inventory", "args": {"item": "红豆沙"}}
            print(f"  Thought #{step}: 需要查库存 → Action: check_inventory(红豆沙)")
            try:
                result = check_inventory(**th["args"])
            except TimeoutError as e:
                failures += 1
                print(f"  Observe  : ❌ TimeoutError({e})")
                if failures >= 2:
                    print(f"  Thought  : 连续 {failures} 次失败，放弃该路径 → 降级")
                    return self.degrade(task)
            except ValueError as e:
                print(f"  Observe  : ❌ 参数错误({e}) → 不重试，直接反馈修正")
                return f"⚠️ 请检查参数: {e}"
        return self.degrade(task)

    def degrade(self, task):
        return "😅 库存服务暂时不可用。降级方案：门店电话 400-000-0000 人工查询，或稍后再试。"

print("═" * 56)
print("❌ 没有保护（while True 版）：")
print("  Thought #1..#∞: 查库存 → Timeout → 查库存 → Timeout → ...")
print("  （真实后果：用户等到超时、账单烧穿、服务雪上加霜）")
print("═" * 56)
print("✅ 有 max_steps + 降级：")
guarded = GuardedReActAgent({"check_inventory": check_inventory}, max_steps=4)
print(f"\n💬 最终回复: {guarded.run('红豆沙还有吗？')}")

## 实验 4（Day4 复习）：自洽性投票——多次采样取多数为什么更准

把 Day 4 的评测概念也复习一下：同一个模型、同一个 prompt，采样 k 次对答案投票。
用蒙特卡洛模拟一个"单次正确率 p=0.6"的解题器，看多数投票的准确率如何随 k 上升。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
P_CORRECT, N_PROBLEMS = 0.6, 20000

ks = np.arange(1, 16, 2)          # 奇数次避免平票: 1,3,5,...,15
accuracies = []
for k in ks:
    # N_PROBLEMS 道二值题，每题采样 k 次，每次以 p 独立答对
    samples = rng.random((N_PROBLEMS, k)) < P_CORRECT
    votes_correct = samples.sum(axis=1) > k / 2     # 多数票 = 答对
    accuracies.append(votes_correct.mean())

for k, acc in zip(ks, accuracies):
    bar = "█" * int((acc - 0.5) * 100)
    print(f"k={k:>2d} 次投票: 准确率 {acc:.3f} {bar}")

print(f"\n单次采样 (k=1): {accuracies[0]:.3f}")
print(f"多数投票 (k=15): {accuracies[-1]:.3f}（提升 {(accuracies[-1]-accuracies[0])*100:.1f} 个百分点）")
print("\n前提条件：错误要『分散』（独立采样）。如果错误是系统性的，投一万次也是错。")
print("代价：k 次投票 = k 倍 token 账单 —— 准确率与成本永远在博弈。")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(ks, accuracies, "o-", color="#FF6B6B", label="多数投票准确率")
ax.axhline(P_CORRECT, color="gray", ls="--", label=f"单次采样基线 p={P_CORRECT}")
for k, acc in zip(ks, accuracies):
    ax.annotate(f"{acc:.2f}", (k, acc), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9)
ax.set_xlabel("投票次数 k（奇数，避免平票）")
ax.set_ylabel("准确率")
ax.set_title("自洽性投票：错误相互独立时，多数票显著优于单次采样")
ax.set_xticks(ks)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 复习自检结论

| 实验 | 覆盖的知识点 | 来自 |
|---|---|---|
| 1 | ReAct 状态机：Thought→Action→Observation 循环 + 终止条件 | Day 1 / Day 3 |
| 2 | 多跳任务必须有循环结构；单轮直答结构性缺信息 | Day 3 |
| 3 | max_steps 防死循环、暂时性/参数错误区别对待、优雅降级 | Day 5 |
| 4 | 自洽性投票：多数票提升准确率的前提与代价 | Day 4 |
| （工具注册、Schema、参数修复回路） | 见 Day 6 notebook 实验 1-4 | Day 2 / Day 6 |

**一周一句话**：Agent = LLM + Tools + Memory + ReAct 循环——让 AI 从"只会说"变成"能做事"。

→ 综合测试题、知识全景图、面试高频题：见同名 md 第二、三、七节。